In [ ]:
import pandas as pd
import numpy as np

import copy

from sklearn.preprocessing import StandardScaler

import math
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [8]:
df_market = pd.read_csv('../data/preprocessed/model_data.csv')

In [9]:
target_col = "future_rv_20d"

spy_features = [
    "return",
    "log_return",
    "return_5d",
    "return_20d",
    "rv_5d",
    "rv_20d",
    "rv_60d",
    "drawdown",
    "spy_volume",
    "spy_avg_vol_20d"
]

vix_features = [
    "vix_close",
    "vix_pct_change",
    "vix_return_5d",
    "vix_rv_20d",
    "spread",
    "rolling_corr_20d"
]

options_features = [
    # Implied volatility levels
    "atm_iv_14d",
    "atm_iv_30d",
    "atm_iv_60d",
    "atm_iv_90d",

    # Volatility skew
    "rr_25_14d",
    "rr_25_30d",
    "rr_25_60d",
    "rr_25_90d",

    # Volatility smile curvature
    "bf_25_14d",
    "bf_25_30d",
    "bf_25_60d",

    # Term structure
    "atm_term_slope_30_90",

    # IV dynamics
    "atm_iv_30d_change_5d"
]

feature_groups = {
    "SPY": spy_features,
    "SPY + VIX": spy_features + vix_features,
    "SPY + Options": spy_features + options_features,
    "All Features": (
        spy_features + vix_features + options_features
    )
}

In [10]:
LOOKBACK = 60
HORIZON = 20
TARGET = target_col

features = feature_groups["SPY + VIX"]

df = (
    df_market
    .sort_values("date")
    .reset_index(drop=True)
    .copy()
)

df = df.dropna(subset=features + [TARGET]).copy()

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

scaler = StandardScaler()
scaler.fit(df.iloc[:train_end][features])

X_all = scaler.transform(df[features]).astype(
    np.float32
)

y_all = df[TARGET].to_numpy(dtype=np.float32)

In [13]:
def create_sequences(X, y, lookback):
    X_seq = []
    y_seq = []
    origins = []
    for t in range(lookback-1, len(X)):
        X_seq.append(X[t-lookback+1:t+1])
        y_seq.append(y[t])
        origins.append(t)

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(y_seq, dtype=np.float32),
        np.asarray(origins)
    )

In [14]:
X_seq, y_seq, origins = create_sequences(X_all, y_all, LOOKBACK)

In [15]:
train_mask = origins + HORIZON < train_end

val_mask = (
    (origins >= train_end) &
    (origins + HORIZON < val_end)
)

test_mask = origins >= val_end

X_train = X_seq[train_mask]
y_train = y_seq[train_mask]

X_val = X_seq[val_mask]
y_val = y_seq[val_mask]

X_test = X_seq[test_mask]
y_test = y_seq[test_mask]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (2049, 60, 16) (2049,)
Validation: (436, 60, 16) (436,)
Test: (457, 60, 16) (457,)


In [18]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)

        pe[:, 1::2] = torch.cos(div_term * position)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
class VolatilityTransformer(nn.Module):
    def __init__(
            self,
            num_features,
            d_model=64,
            nhead=4,
            num_layers=2,
            dim_feedforward=128,
            dropout=0.2
    ):
        super().__init__()

        self.input_projection = nn.Linear(num_features, d_model)

        self.pos_encoder = PositionalEncoding(d_model=d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=True
        )

        self.norm = nn.LayerNorm(d_model)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.input_projection(x)

        x = self.pos_encoder(x)

        x = self.transformer(x)

        #Rep of most recent day
        x = x[:, -1, :]

        x = self.norm(x)

        output = self.regressor(x)

        return output.squeeze(-1)

In [22]:
train = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train)
)

val = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(y_val)
)

test = TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(y_test)
)

train_loader = DataLoader(
    train,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test,
    batch_size=32,
    shuffle=False
)

In [25]:
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu'
)

model = VolatilityTransformer(
    d_model=64,
    nhead=4,
    num_features=X_train.shape[-1],
    num_layers=2,
    dim_feedforward=128,
    dropout=0.2
).to(device)

criterion = nn.L1Loss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)

/var/folders/mz/yp0n9b4n6yqf6qq86zgmmnqw0000gn/T/ipykernel_54567/708484734.py:27: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


In [ ]:
def train_model(
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        device,
        epochs=100,
        patience=15
):
    history = {
        'train_mae': [],
        'val_mae': []
    }

    best_val = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_err = 0.0
        train_count = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            predictions = model(X_batch)

            loss = criterion(predictions, y_batch)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            train_err += (loss.item() * len(y_batch))

            train_count += len(y_batch)

        train_mae = train_err / train_count

        model.eval()
        val_err = 0.0
        val_count = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                predictions = model(X_batch)

                loss = criterion(predictions, y_batch)

                val_err += loss.item() * len(y_batch)

                val_count += len(y_batch)

        val_mae = val_err/ val_count

        history["train_mae"].append(train_mae)
        history["val_mae"].append(val_mae)

        if val_mae < best_val:
            best_val = val_mae
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience_counter += 1

        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1:03d} | "
                f"Train MAE: {train_mae:.6f} | "
                f"Val MAE: {val_mae:.6f}"
            )

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)

    return model, history